<a href="https://colab.research.google.com/github/0xfffddd/Coding/blob/main/687Final_partb_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Part B. Multimodal LLM Feature Engineering with Gemini
# Checkpoint and cache feature were added in case interrupted

!pip -q install google-generativeai pandas pillow tqdm
import os, re, json, time, hashlib
import pandas as pd
from PIL import Image
from tqdm import tqdm
import google.generativeai as genai
import getpass

# 0. Paths
CSV_PATH = "/content/dataset_history.csv"
ZIP_PATH = "/content/image.zip"
IMG_ROOT = "/content/images"
CACHE_DIR = "/content/llm_cache"
CHECKPOINT_PATH = "/content/_checkpoint_features.csv"
FINAL_OUT = "/content/dataset_history_with_llm_features.csv"
os.makedirs(CACHE_DIR, exist_ok=True)

# 1. API Key
api_key = getpass.getpass("Enter your Gemini API Key: ")
genai.configure(api_key=api_key)
# 2. Model selection
preferred = [
    "models/gemini-2.0-flash",
    "models/gemini-2.5-flash",
    "models/gemini-flash-latest",
    "models/gemini-2.0-flash-lite",
    "models/gemini-flash-lite-latest",
]
available = {m.name for m in genai.list_models() if "generateContent" in m.supported_generation_methods}
model_name = next((m for m in preferred if m in available), sorted(list(available))[0])
print("Chosen model:", model_name)
model = genai.GenerativeModel(model_name)

# 3 Unzip images
if os.path.exists(ZIP_PATH):
    !unzip -q -o /content/image.zip -d /content/
else:
    print("Warning: image.zip not found. If you rely on local images, upload it.")


# 4. Load dataset

df = pd.read_csv(CSV_PATH)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head(3))

# 5. Choose image source: prefer local image_path if valid
def normalize_path(p: str) -> str:
    return str(p).replace("\\", "/").strip()
img_mode = None
if "image_path" in df.columns:
    test_path = os.path.join(IMG_ROOT, normalize_path(df["image_path"].iloc[0]))
    if os.path.exists(test_path):
        img_mode = "local"
        img_col = "image_path"
        print("Using LOCAL images from column:", img_col)
    else:
        print("image_path exists but first file not found:", test_path)

if img_mode is None:
    # fallback: use URL column "image" if exists
    if "image" in df.columns:
        img_mode = "url"
        img_col = "image"
        print("Falling back to URL images from column:", img_col)
    else:
        raise ValueError("No usable image source found (need image_path or image URL).")

# 6. Feature keys+prompt
FEATURE_KEYS = [
    "image_clarity",
    "style_appeal",
    "trend_alignment",
    "perceived_quality",
    "versatility",
    "price_reasonableness",
    "purchase_likelihood"
]

PROMPT = f"""
You are an experienced fashion e-commerce buyer working under a fixed advertising budget.
Rate how likely this product is to generate at least one order.

Scoring: use INTEGER 1-5 only (1=very poor, 3=average/uncertain, 5=excellent).
If uncertain, use 3. Do NOT output explanations.

Return ONLY a valid JSON object with EXACTLY these keys:
{FEATURE_KEYS}

Definitions:
image_clarity: photo quality and product focus
style_appeal: visual attractiveness of style
trend_alignment: how on-trend the item is
perceived_quality: perceived fabric/build quality
versatility: ease to match / broad usability
price_reasonableness: value for money given title+price+image
purchase_likelihood: overall likelihood of at least one order
"""

# 7. image load + resize (reduce compute/quota)
def load_local_image(rel_path: str, max_side: int = 448) -> Image.Image:
    full = os.path.join(IMG_ROOT, normalize_path(rel_path))
    img = Image.open(full).convert("RGB")
    w, h = img.size
    scale = min(max_side / max(w, h), 1.0)
    if scale < 1.0:
        img = img.resize((int(w*scale), int(h*scale)))
    return img

# 8. JSON parsing
def parse_json_strict(text: str) -> dict:
    t = (text or "").strip()
    t = re.sub(r"```json|```", "", t).strip()
    m = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if m:
        t = m.group(0).strip()
    obj = json.loads(t)
    out = {}
    for k in FEATURE_KEYS:
        v = obj.get(k, None)
        try:
            iv = int(v)
            iv = max(1, min(5, iv))
        except:
            iv = None
        out[k] = iv
    return out

# 9. Caching
def cache_key(row_id, title, price, img_ref) -> str:
    raw = f"{model_name}|{row_id}|{title}|{price}|{img_ref}"
    return hashlib.md5(raw.encode("utf-8")).hexdigest()
def cache_get(key):
    fp = os.path.join(CACHE_DIR, f"{key}.json")
    if os.path.exists(fp):
        with open(fp, "r", encoding="utf-8") as f:
            return json.load(f)
    return None
def cache_set(key, obj):
    fp = os.path.join(CACHE_DIR, f"{key}.json")
    with open(fp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

# 10. Adaptive throttling to avoid error 429 (request too fast)
class AdaptiveRateLimiter:
    """
    Starts with a base delay between calls.
    On 429, increases delay aggressively and holds it for a while.
    On success streak, slowly decreases delay.
    """
    def __init__(self, base_delay=2.0, max_delay=60.0, min_delay=1.0):
        self.delay = base_delay
        self.max_delay = max_delay
        self.min_delay = min_delay
        self.success_streak = 0
        self.last_was_429 = False
    def on_success(self):
        self.success_streak += 1
        self.last_was_429 = False
        # after several successes, gently reduce delay
        if self.success_streak >= 5:
            self.delay = max(self.min_delay, self.delay * 0.85)
            self.success_streak = 0
    def on_429(self):
        self.success_streak = 0
        self.last_was_429 = True
        # increase delay quickly
        self.delay = min(self.max_delay, max(self.delay * 1.8, self.delay + 5))
    def sleep(self):
        time.sleep(self.delay)
limiter = AdaptiveRateLimiter(base_delay=3.0, min_delay=1.5, max_delay=60.0)

# 11. Resume from checkpoint if exists
start_idx = 0
existing = None
if os.path.exists(CHECKPOINT_PATH):
    try:
        existing = pd.read_csv(CHECKPOINT_PATH)
        start_idx = len(existing)
        print(f"Resuming from checkpoint: {CHECKPOINT_PATH} (rows={start_idx})")
    except Exception as e:
        print("Could not read checkpoint, starting fresh. Reason:", e)
results = []
if existing is not None and start_idx > 0:
    # If checkpoint already has feature columns, reuse
    keep_cols = [c for c in existing.columns if c in FEATURE_KEYS or c == "_error"]
    results = existing[keep_cols].to_dict(orient="records")

# 12. Extract with retries + 429 handling
def extract_one(i, row, max_retries=6):
    title = str(row.get("title", ""))
    price = row.get("price", "")
    img_ref = row[img_col]
    key = cache_key(i, title, price, img_ref)
    cached = cache_get(key)
    if cached is not None:
        return cached, False
    # local image only
    img = load_local_image(img_ref)
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = model.generate_content(
                [PROMPT + f"\nTitle: {title}\nPrice: {price}", img],
                generation_config={"temperature": 0, "max_output_tokens": 120}
            )
            feats = parse_json_strict(resp.text)
            feats["_error"] = None
            cache_set(key, feats)
            limiter.on_success()
            return feats, True
        except Exception as e:
            last_err = str(e)
            if "429" in last_err:
                limiter.on_429()
            # exponential backoff per attempt, plus limiter delay
            backoff = min(60, (2 ** attempt))
            time.sleep(backoff)
            limiter.sleep()
    fail = {k: None for k in FEATURE_KEYS}
    fail["_error"] = last_err
    cache_set(key, fail)
    return fail, True


# 13. Run loop

SAVE_EVERY = 10
MAX_CALLS = len(df)
fresh_calls = 0
for i in tqdm(range(start_idx, min(len(df), MAX_CALLS))):
    row = df.iloc[i]
    feats, did_call = extract_one(i, row, max_retries=6)
    if did_call:
        fresh_calls += 1
    results.append({k: feats.get(k) for k in FEATURE_KEYS} | {"_error": feats.get("_error")})
    # checkpoint
    if (i + 1) % SAVE_EVERY == 0:
        tmp = pd.concat([df.iloc[:len(results)].reset_index(drop=True),
                         pd.DataFrame(results)], axis=1)
        tmp.to_csv(CHECKPOINT_PATH, index=False)

# final save
features_df = pd.DataFrame(results)
out_df = pd.concat([df.iloc[:len(results)].reset_index(drop=True), features_df], axis=1)
out_df.to_csv(FINAL_OUT, index=False)
print("Saved final:", FINAL_OUT)
print("Fresh API calls made in this run:", fresh_calls)
print("Error rate:", out_df["_error"].notna().mean())
display(out_df.head(5))